In [ ]:
import os
import numpy as np
from collections import Counter
from multiprocessing import Pool, cpu_count

def contar_clases_un_archivo(args):
    """Lee un archivo _Y.npy y cuenta cuántos píxeles tiene de cada clase."""
    carpeta, archivo = args
    try:
        # Cargar la máscara de etiquetas
        ruta_completa = os.path.join(carpeta, archivo)
        parche_y = np.load(ruta_completa)
        
        # Obtener los valores únicos y sus frecuencias en este parche
        valores, frecuencias = np.unique(parche_y, return_counts=True)
        
        # Devolver un diccionario con los conteos {clase: cantidad_pixeles}
        return dict(zip(valores, frecuencias))
    except Exception as e:
        return {}

def obtener_value_counts_npy(carpeta_dataset):
    # 1. Buscar todos los archivos de etiquetas _Y.npy
    archivos_Y = [f for f in os.listdir(carpeta_dataset) if f.endswith('_Y.npy')]
    total_archivos = len(archivos_Y)
    
    if total_archivos == 0:
        print(f"No se encontraron archivos _Y.npy en {carpeta_dataset}")
        return
        
    print(f"Analizando {total_archivos} archivos _Y.npy en paralelo...")
    
    # 2. Preparar los argumentos para la paralelización
    num_processes = cpu_count() # <--- Aquí se define
    argumentos = [(carpeta_dataset, f) for f in archivos_Y]
    
    contador_global = Counter()
    
    # 3. Ejecutar el conteo en paralelo (CORREGIDO 'num_processes')
    with Pool(num_processes) as pool: # <--- Usamos el mismo nombre exacto
        # Usamos imap_unordered para ir procesando los resultados conforme terminen
        for conteo_parcial in pool.imap_unordered(contar_clases_un_archivo, argumentos, chunksize=100):
            contador_global.update(conteo_parcial)
            
    # 4. Mostrar los resultados ordenados por la clase (código)
    print("\n" + "="*40)
    print("      CONTEO DE PÍXELES POR CLASE")
    print("="*40)
    print(f"{'Clase':<10}{'Número de Píxeles':<20}")
    print("-"*40)
    
    for clase in sorted(contador_global.keys()):
        print(f"{clase:<10}{contador_global[clase]:<20,}")
    print("="*40)
    
    return contador_global

# --- Ejecución ---
if __name__ == '__main__':
    carpeta_tfg = r"D:\TFG\dataset_parches_50x50"
    conteos = obtener_value_counts_npy(carpeta_tfg)

Analizando 1176897 archivos _Y.npy en paralelo...
